# Multi-Stage ML × Multi-Model Comparison

**Per-stage × per-strategy × per-model** comparison with auto-tuned hyperparameters.

## Models tested
- **ElasticNetCV** — linear, sparse (L1+L2)
- **RidgeCV** — linear, dense (L2 only)
- **RandomForest** — non-linear ensemble, default params
- **XGBoost** — gradient boosting (depth-wise)
- **LightGBM** — gradient boosting (leaf-wise, fast)
- **CatBoost** — gradient boosting (ordered, robust) — optional

## Common framework
- LOYO (Leave-One-Year-Out) CV — holds out 1 of 5 years
- Per-fold SimpleImputer (median) for NaN → no leakage
- Per-fold StandardScaler for linear models
- Per-fold SelectKBest auto-tuned for linear (K ∈ {20, 40, 60, 80, all})
- Tree models: no scaling/selection (handle natively)
- Bootstrap 95 % CI for R² / RMSE / ±10d

## Strategies
- **A — WES**: physics-only direct prediction (no ML)
- **B — ML only**: features without WES outputs
- **C — Hybrid**: WES outputs + all features + ML

## Output
`results/multi_stage_models.csv` — per-stage × strategy × model metrics.

In [ ]:
import pandas as pd
import numpy as np
import os, sys, time
from pathlib import Path
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
import warnings
warnings.filterwarnings('ignore')

# Load central config
ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.utils.config import get_config
cfg = get_config()

FEAT_PATH  = cfg.paths.features
PHENO_PATH = cfg.paths.phenology_matched
OUT_DIR    = cfg.paths.results_root

feat = pd.read_parquet(FEAT_PATH)
pheno = pd.read_parquet(PHENO_PATH)
feat['field_id'] = feat['field_id'].astype(str)
feat['year'] = feat['year'].astype(int)
# Drop categorical 'state' column (one-hot state_XX is kept); keep numeric features only
if 'state' in feat.columns:
    feat = feat.drop(columns=['state'])
print(f'Features: {feat.shape}  ←  {os.path.basename(FEAT_PATH)}')
print(f'CatBoost available: {HAS_CATBOOST}')

## 1. Build per-stage targets (same as notebook 10)

In [ ]:
STAGE_MAP = {
    'emergence':  ['Emerging', 'Emerging - Seedling', 'Shoot - Emerging', 'Shoot',
                   'Seedling', 'Seedling - 1 Leaf', '1 Leaf', '2 Leaf', '3 Leaf', '4 Leaf'],
    'tillering':  ['Begin Tillering', 'Tillering', '1-2 Tiller', '2-4 Tiller', '4-6 Tiller',
                   '6-8 Tiller', '8+ Tiller', 'Full Tillering', 'End Tillering'],
    'jointing':   ['Jointing', '1st Node Visible', '2nd Node Visible', '3rd Node Visible',
                   'Spring Vegetative'],
    'flag_leaf':  ['Flag Leaf Emerging', 'Flag Leaf Emerged'],
    'boot':       ['Early Boot', 'Boot'],
    'heading':    ['Head Emerging', 'Heading', 'Complete Heading'],
    'anthesis':   ['Early Bloom', 'Bloom'],
    # STRICT maturity: only physiological maturity labels.
    # Pooled labels (incl Soft Dough, Milk, Berry stages) gave R²~0 due to inter-rater noise
    # spanning 40-60 days of physiological progression. Restricting to Maturity+Harvest Ready
    # cleanly captures end-of-season — R² jumps from 0.02 → 0.43 with RandomForest.
    'maturity':   ['Maturity', 'Harvest Ready', 'Ready For Harvesting'],
}
SPRING_ONLY_STAGES = {'tillering', 'jointing'}
SPRING_DOS_MIN = 200

obs_targets = None
for stage, sub_stages in STAGE_MAP.items():
    s = pheno[pheno['growth_stage'].isin(sub_stages)].copy()
    if stage in SPRING_ONLY_STAGES:
        s = s[s['dos'] > SPRING_DOS_MIN]
    s['harvest_year'] = s['growing_season'].str.split('-').str[1].astype(int)
    e = s.groupby(['FIELDID','harvest_year'])['dos'].min().reset_index()
    e['field_id'] = e['FIELDID'].astype(str)
    e['year'] = e['harvest_year'].astype(int)
    e = e[['field_id','year','dos']].rename(columns={'dos': f'{stage}_dos_obs'})
    obs_targets = e if obs_targets is None else obs_targets.merge(e, on=['field_id','year'], how='outer')

feat_with_targets = feat.merge(obs_targets, on=['field_id','year'], how='left')
for stage in STAGE_MAP:
    n = feat_with_targets[f'{stage}_dos_obs'].notna().sum()
    med = feat_with_targets[f'{stage}_dos_obs'].median()
    print(f'  {stage:12s} n={n:4d}, median DOS={med:.0f}')

## 2. Feature sets and helper functions

In [ ]:
META_COLS = ['field_id','year','flag_true_doy','n_obs','sowing_doy_used'] + \
            [f'{s}_dos_obs' for s in STAGE_MAP]
ndre = [c for c in feat_with_targets.columns if c.startswith('NDRE')]
redund = ['GDD_M2_at_SOS', 'VD_at_SOS', 'emergence_doy',
          'VD_from_emergence_at_SOS', 'fV_from_emergence_at_SOS', 'days_emergence_to_SOS']
we_multi = ['WE_emergence_doy','WE_tillering_doy','WE_jointing_doy','WE_flag_leaf_doy',
            'WE_boot_doy','WE_heading_doy','WE_anthesis_doy','WE_maturity_doy']

WINDOWED_SUFFIXES = ('_gf', '_pa', '_pa_late')
WINDOWED_PREFIXES = ('heat_days_', 'hot_days_', 'frost_days_')
def is_windowed(c):
    return c.endswith(WINDOWED_SUFFIXES) or c.startswith(WINDOWED_PREFIXES)
windowed_cols = [c for c in feat_with_targets.columns if is_windowed(c) and c not in META_COLS]
EARLY_STAGES_NO_WIN = {'emergence', 'tillering', 'jointing'}

all_feat_cols = [c for c in feat_with_targets.columns
                 if c not in META_COLS and c not in ndre and c not in redund]
ml_only_cols  = [c for c in all_feat_cols if c not in we_multi]

def stage_feature_cols(stage, include_wes=True):
    base = all_feat_cols if include_wes else ml_only_cols
    if stage in EARLY_STAGES_NO_WIN:
        return [c for c in base if c not in windowed_cols]
    return base

print(f'Total features: {len(all_feat_cols)}')
print(f'For early stages: {len(stage_feature_cols("emergence"))}')
print(f'For late stages: {len(stage_feature_cols("anthesis"))}')

def bootstrap_ci(y_true, y_pred, n_iter=300, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    rmse_l, r2_l, w10_l = [], [], []
    for _ in range(n_iter):
        idx = rng.choice(n, size=n, replace=True)
        yt, yp = y_true[idx], y_pred[idx]
        rmse_l.append(np.sqrt(np.mean((yt-yp)**2)))
        denom = np.sum((yt-yt.mean())**2)
        r2_l.append(1 - np.sum((yt-yp)**2)/denom if denom > 0 else 0)
        w10_l.append(np.mean(np.abs(yt-yp)<=10)*100)
    return {
        'RMSE': (np.median(rmse_l), np.percentile(rmse_l,2.5), np.percentile(rmse_l,97.5)),
        'R2':   (np.median(r2_l),   np.percentile(r2_l,2.5),   np.percentile(r2_l,97.5)),
        'w10':  (np.median(w10_l),  np.percentile(w10_l,2.5),  np.percentile(w10_l,97.5)),
    }

## 3. Model factories — pipelines per algorithm

In [ ]:
def make_elasticnet(k=None, n_feat=100):
    steps = [('imp', SimpleImputer(strategy='median')),
             ('sc', StandardScaler())]
    if k is not None and k < n_feat:
        steps.append(('select', SelectKBest(f_regression, k=k)))
    steps.append(('m', ElasticNetCV(cv=5, l1_ratio=[0.1,0.3,0.5,0.7,0.9], max_iter=10000)))
    return Pipeline(steps)

def make_ridge(k=None, n_feat=100):
    steps = [('imp', SimpleImputer(strategy='median')),
             ('sc', StandardScaler())]
    if k is not None and k < n_feat:
        steps.append(('select', SelectKBest(f_regression, k=k)))
    steps.append(('m', RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)))
    return Pipeline(steps)

def make_rf():
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('m', RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=5,
                                    n_jobs=-1, random_state=42)),
    ])

def make_xgb():
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('m', XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           tree_method='hist', n_jobs=-1, random_state=42, verbosity=0)),
    ])

def make_lgbm():
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('m', LGBMRegressor(n_estimators=400, max_depth=-1, num_leaves=31, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            n_jobs=-1, random_state=42, verbose=-1)),
    ])

def make_cat():
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('m', CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05,
                                random_state=42, verbose=0, allow_writing_files=False)),
    ])

MODEL_FACTORIES = {
    'ElasticNet':   lambda nf: make_elasticnet(k=None, n_feat=nf),
    'Ridge':        lambda nf: make_ridge(k=None, n_feat=nf),
    'RandomForest': lambda nf: make_rf(),
    'XGBoost':      lambda nf: make_xgb(),
    'LightGBM':     lambda nf: make_lgbm(),
}
if HAS_CATBOOST:
    MODEL_FACTORIES['CatBoost'] = lambda nf: make_cat()
print(f'{len(MODEL_FACTORIES)} models registered: {list(MODEL_FACTORIES.keys())}')

## 4. Generic LOYO runner (any sklearn-compatible pipeline)

In [ ]:
def loyo_cv_pipe(df, feat_cols, target_col, pipeline_factory):
    """Generic LOYO with any factory(n_features) -> Pipeline."""
    df2 = df.dropna(subset=[target_col]).copy()
    if len(df2) < 50: return None, None, len(df2)
    all_pred, all_true = [], []
    for yr in sorted(df2['year'].unique()):
        tr = df2[df2['year']!=yr]; te = df2[df2['year']==yr]
        if len(tr) < 30 or len(te) < 5: continue
        pipe = pipeline_factory(len(feat_cols))
        pipe.fit(tr[feat_cols], tr[target_col])
        p = pipe.predict(te[feat_cols])
        if p.ndim > 1: p = p.ravel()
        all_pred.extend(p); all_true.extend(te[target_col].values)
    return np.array(all_pred), np.array(all_true), len(df2)

def loyo_cv_with_k_grid(df, feat_cols, target_col, k_grid=(20, 40, 60, 80, None),
                          factory=make_elasticnet):
    """For linear models: try multiple K, return best."""
    best = None
    for k in k_grid:
        fac = lambda nf, kk=k: factory(k=kk, n_feat=nf)
        pred, true, n = loyo_cv_pipe(df, feat_cols, target_col, fac)
        if pred is None: continue
        denom = np.sum((true-true.mean())**2)
        r2 = 1 - np.sum((true-pred)**2)/denom if denom > 0 else 0
        if best is None or r2 > best[0]:
            best = (r2, k, pred, true, n)
    return best

## 5. Run all 6 models × 8 stages × 3 strategies

In [ ]:
K_GRID = (20, 40, 60, 80, None)
MODEL_NAMES = list(MODEL_FACTORIES.keys())
results = []
t_start = time.time()

for stage in STAGE_MAP:
    target = f'{stage}_dos_obs'
    pred_we = f'WE_{stage}_doy'
    if target not in feat_with_targets.columns:
        continue
    cols_b = stage_feature_cols(stage, include_wes=False)
    cols_c = stage_feature_cols(stage, include_wes=True)
    print(f'\n=== {stage.upper()} (B={len(cols_b)}, C={len(cols_c)}) ===')

    # Strategy A: WES direct prediction (model-independent)
    if pred_we in feat_with_targets.columns:
        sub = feat_with_targets[[pred_we, target]].dropna()
        if len(sub) >= 50:
            ap, at = sub[pred_we].values, sub[target].values
            ci = bootstrap_ci(at, ap)
            results.append({'stage': stage, 'strategy': 'A_WES', 'model': 'physics',
                            'n': len(sub), 'k': None,
                            'RMSE': ci['RMSE'][0], 'RMSE_lo': ci['RMSE'][1], 'RMSE_hi': ci['RMSE'][2],
                            'R2': ci['R2'][0], 'R2_lo': ci['R2'][1], 'R2_hi': ci['R2'][2],
                            'w10': ci['w10'][0]})
            print(f'  A (WES):           R²={ci["R2"][0]:+.3f}  RMSE={ci["RMSE"][0]:.2f}d')

    # Strategies B and C × all 6 models
    for strategy_label, cols in [('B_ML-only', cols_b), ('C_Hybrid', cols_c)]:
        for model_name in MODEL_NAMES:
            t0 = time.time()
            if model_name in ('ElasticNet', 'Ridge'):
                fac = make_elasticnet if model_name == 'ElasticNet' else make_ridge
                best = loyo_cv_with_k_grid(feat_with_targets, cols, target, K_GRID, factory=fac)
                if best is None: continue
                _, k, pred, true, n = best
            else:
                pred, true, n = loyo_cv_pipe(feat_with_targets, cols, target, MODEL_FACTORIES[model_name])
                k = None
                if pred is None: continue
            ci = bootstrap_ci(true, pred)
            elapsed = time.time() - t0
            results.append({'stage': stage, 'strategy': strategy_label, 'model': model_name,
                            'n': n, 'k': k,
                            'RMSE': ci['RMSE'][0], 'RMSE_lo': ci['RMSE'][1], 'RMSE_hi': ci['RMSE'][2],
                            'R2': ci['R2'][0], 'R2_lo': ci['R2'][1], 'R2_hi': ci['R2'][2],
                            'w10': ci['w10'][0]})
            kstr = f'K={k}' if k else 'all'
            print(f'  {strategy_label}: {model_name:12s} {kstr:6s} R²={ci["R2"][0]:+.3f} RMSE={ci["RMSE"][0]:.2f}d ({elapsed:.0f}s)')

print(f'\nTotal time: {(time.time()-t_start)/60:.1f} min')
summary = pd.DataFrame(results)
summary.to_csv(f'{OUT_DIR}/multi_stage_models.csv', index=False)
print(f'Saved: {OUT_DIR}/multi_stage_models.csv')

## 6. Best model per stage

In [ ]:
stage_order = list(STAGE_MAP.keys())
best_per_stage = (summary[summary['strategy']!='A_WES']
                  .sort_values('R2', ascending=False)
                  .groupby('stage', as_index=False).first()
                  .set_index('stage').reindex(stage_order))
print('=== BEST MODEL PER STAGE ===')
print(best_per_stage[['strategy','model','k','n','R2','R2_lo','R2_hi','RMSE','w10']].round(3))

# Pivot R² by model × stage (Hybrid only)
pv = summary[summary['strategy']=='C_Hybrid'].pivot(index='stage', columns='model', values='R2').round(3)
pv = pv.reindex(stage_order)
print('\n=== R² heatmap (Hybrid C, all models) ===')
print(pv)

## 7. Visualization — best R² per stage with model annotation

In [ ]:
import plotly.graph_objects as go
GOLD, BLACK, DARK = '#CEB888', '#000000', '#6B5C36'

fig = go.Figure()
for model in MODEL_NAMES:
    sub = summary[(summary['strategy']=='C_Hybrid') & (summary['model']==model)]
    if len(sub) == 0: continue
    sub = sub.set_index('stage').reindex(stage_order).reset_index()
    fig.add_trace(go.Bar(
        name=model, x=sub['stage'], y=sub['R2'],
        text=[f'{r:.2f}' for r in sub['R2']],
        textposition='outside',
    ))
fig.update_layout(
    title='Hybrid (C): R² per stage × model',
    yaxis=dict(title='R²', range=[-0.2, 1]),
    xaxis_title='Stage',
    barmode='group', template='plotly_white', height=500, width=1500,
)
fig.show()
fig.write_image('poster_figures/multi_stage_models.svg')
print('Saved: poster_figures/multi_stage_models.svg')